# MedGemma QLoRA notebook-first

Este notebook es separado del baseline. La idea es probar QLoRA de forma visible, celda por celda, antes de convertirlo en código modular.

Objetivos:

1. Entrenar un adapter QLoRA pequeño usando ACRIMA como tarea de clasificación `glaucoma` vs `non_glaucoma`.
2. Evaluar el adapter en imágenes no vistas del mismo dataset.
3. Dejar preparado el camino para QLoRA de descripción con PubMed-Ophtha mientras llega el dataset oficial.

Nota: ACRIMA solo trae etiquetas de clase, no descripciones expertas. Por eso aquí la parte medible es clasificación. La parte de descripción queda como experimento puente.

In [ ]:
REPO_URL = "https://github.com/Luco1421/utils_medgemma.git"
REPO_DIR = "/content/utils_medgemma"
MODEL_ID = "google/medgemma-1.5-4b-it"

## 1. Setup rápido

In [ ]:
import os, shutil

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

!git clone "{REPO_URL}" "{REPO_DIR}"
%cd "{REPO_DIR}"
!pip install -q -r requirements-colab.txt

## 2. Imports, token y dataset

In [ ]:
import glob
import json
import random
import re
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import torch
from PIL import Image
from google.colab import userdata
from huggingface_hub import HfApi

from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig,
)
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

HF_TOKEN = userdata.get("HF_TOKEN")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

api = HfApi()
print(api.whoami(token=HF_TOKEN))
print(api.model_info(MODEL_ID, token=HF_TOKEN).modelId)

DEVICE = "cuda"
assert torch.cuda.is_available(), "Activa GPU en Colab antes de correr QLoRA."
print(torch.cuda.get_device_name(0))

In [ ]:
glaucoma_images = sorted(glob.glob("dataset/Glaucoma/*"))
normal_images = sorted(glob.glob("dataset/Non Glaucoma/*"))

print("Glaucoma:", len(glaucoma_images))
print("Non Glaucoma:", len(normal_images))

TEST_IMAGE = glaucoma_images[0] if glaucoma_images else normal_images[0]
TEST_IMAGE

## 3. Construir splits pequeños para QLoRA

Para un smoke test medible:

- entrenamos con pocas imágenes por clase;
- evaluamos en imágenes distintas;
- la respuesta esperada es solo `glaucoma` o `non_glaucoma`.

Luego puedes subir `TRAIN_PER_CLASS` y `EVAL_PER_CLASS`.

In [ ]:
TRAIN_PER_CLASS = 12
EVAL_PER_CLASS = 8
SPLIT_SEED = 42

CLASSIFICATION_PROMPT = """
You are evaluating a color fundus photograph for glaucoma screening.
Choose exactly one label from this list:
- glaucoma
- non_glaucoma

Reply with only the label and no explanation.
""".strip()

def split_images(images, train_n, eval_n, seed):
    rng = random.Random(seed)
    shuffled = list(images)
    rng.shuffle(shuffled)
    return shuffled[:train_n], shuffled[train_n:train_n + eval_n]

g_train, g_eval = split_images(glaucoma_images, TRAIN_PER_CLASS, EVAL_PER_CLASS, SPLIT_SEED)
n_train, n_eval = split_images(normal_images, TRAIN_PER_CLASS, EVAL_PER_CLASS, SPLIT_SEED)

def make_classification_rows(glaucoma_paths, normal_paths):
    rows = []
    rows.extend({"image": p, "label": "glaucoma"} for p in glaucoma_paths)
    rows.extend({"image": p, "label": "non_glaucoma"} for p in normal_paths)
    random.Random(SPLIT_SEED).shuffle(rows)
    return rows

train_rows = make_classification_rows(g_train, n_train)
eval_rows = make_classification_rows(g_eval, n_eval)

print("train:", len(train_rows), "eval:", len(eval_rows))
train_rows[:3]

## 4. Convertir ejemplos a conversaciones multimodales

In [ ]:
def to_classification_messages(row):
    return {
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": Image.open(row["image"]).convert("RGB")},
                    {"type": "text", "text": CLASSIFICATION_PROMPT},
                ],
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": row["label"]}],
            },
        ]
    }

train_dataset = [to_classification_messages(row) for row in train_rows]
eval_dataset = [to_classification_messages(row) for row in eval_rows]
train_dataset[0]

## 5. Cargar MedGemma en 4-bit + LoRA

Esto es QLoRA: modelo base en 4-bit, adapters LoRA entrenables.

In [ ]:
major, minor = torch.cuda.get_device_capability()
TRAIN_DTYPE = torch.bfloat16 if major >= 8 else torch.float16
print("GPU capability:", (major, minor), "TRAIN_DTYPE:", TRAIN_DTYPE)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=TRAIN_DTYPE,
)

processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    dtype=TRAIN_DTYPE,
    device_map="auto",
    quantization_config=bnb_config,
    token=HF_TOKEN,
)

model.config.use_cache = False
try:
    model.gradient_checkpointing_enable()
except Exception as exc:
    print("gradient_checkpointing_enable no disponible:", exc)

peft_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

print(type(model))

## 6. Collate function

Esta celda prepara texto, imágenes y labels para SFTTrainer.

In [ ]:
def extract_one_image(messages):
    for msg in messages:
        content = msg.get("content", [])
        if not isinstance(content, list):
            content = [content]
        for element in content:
            if isinstance(element, dict) and element.get("type") == "image":
                return element["image"].convert("RGB")
    raise ValueError("No image found in messages")

def collate_fn(examples):
    texts = []
    images = []

    for example in examples:
        messages = example["messages"]
        texts.append(
            processor.apply_chat_template(
                messages,
                add_generation_prompt=False,
                tokenize=False,
            ).strip()
        )
        images.append(extract_one_image(messages))

    batch = processor(
        text=texts,
        images=images,
        return_tensors="pt",
        padding=True,
    )

    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100

    for token_name in ("boi_token_id", "image_token_id", "eoi_token_id"):
        token_id = getattr(processor.tokenizer, token_name, None)
        if token_id is not None:
            labels[labels == token_id] = -100

    batch["labels"] = labels
    return batch

## 7. Entrenar QLoRA

In [ ]:
QLORA_OUTPUT_DIR = "checkpoints/medgemma_qlora_acrima_classifier"

training_args = SFTConfig(
    output_dir=QLORA_OUTPUT_DIR,
    max_steps=20,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    bf16=(TRAIN_DTYPE == torch.bfloat16),
    fp16=(TRAIN_DTYPE == torch.float16),
    logging_steps=1,
    save_steps=20,
    save_total_limit=1,
    remove_unused_columns=False,
    report_to="none",
    dataset_text_field="",
    dataset_kwargs={"skip_prepare_dataset": True},
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    peft_config=peft_config,
    processing_class=processor,
    data_collator=collate_fn,
)

trainer.train()
trainer.save_model(QLORA_OUTPUT_DIR)
processor.save_pretrained(QLORA_OUTPUT_DIR)

model = trainer.model
model.eval()
print("Adapter guardado en:", QLORA_OUTPUT_DIR)

## 8. Helpers de generación y parsing

In [ ]:
def first_model_device(model_obj):
    if hasattr(model_obj, "hf_device_map"):
        for device in model_obj.hf_device_map.values():
            if device not in {"cpu", "disk", "meta"}:
                return device
    if hasattr(model_obj, "device"):
        return model_obj.device
    return next(model_obj.parameters()).device

def generate_with_model(model_obj, processor_obj, prompt, image_path=None, max_new_tokens=32):
    content = []
    if image_path is not None:
        content.append({"type": "image", "image": Image.open(image_path).convert("RGB")})
    content.append({"type": "text", "text": prompt})
    messages = [{"role": "user", "content": content}]

    inputs = processor_obj.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    input_len = inputs["input_ids"].shape[-1]
    device = first_model_device(model_obj)
    for key, value in inputs.items():
        if torch.is_tensor(value):
            if value.is_floating_point():
                inputs[key] = value.to(device=device, dtype=TRAIN_DTYPE)
            else:
                inputs[key] = value.to(device=device)

    with torch.inference_mode():
        output = model_obj.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    generated = output[0][input_len:]
    return processor_obj.decode(generated, skip_special_tokens=True).strip()

def parse_glaucoma_label(text):
    clean = text.lower().strip()
    if clean in {"glaucoma", "non_glaucoma", "non glaucoma", "normal"}:
        return "non_glaucoma" if clean in {"non_glaucoma", "non glaucoma", "normal"} else "glaucoma"
    if re.search(r"non[-_\s]?glaucoma|no glaucoma|not glaucoma|normal", clean):
        return "non_glaucoma"
    if re.search(r"\bglaucoma\b|glaucomatous", clean):
        return "glaucoma"
    return "unknown"

def compute_binary_metrics(results):
    tp = sum(r["true_label"] == "glaucoma" and r["pred_label"] == "glaucoma" for r in results)
    tn = sum(r["true_label"] == "non_glaucoma" and r["pred_label"] == "non_glaucoma" for r in results)
    fp = sum(r["true_label"] == "non_glaucoma" and r["pred_label"] == "glaucoma" for r in results)
    fn = sum(r["true_label"] == "glaucoma" and r["pred_label"] == "non_glaucoma" for r in results)
    unknown = sum(r["pred_label"] == "unknown" for r in results)
    total = len(results)
    known = total - unknown
    return {
        "total": total,
        "known_predictions": known,
        "unknown": unknown,
        "accuracy_known_only": (tp + tn) / known if known else None,
        "accuracy_with_unknown_as_wrong": (tp + tn) / total if total else None,
        "sensitivity_glaucoma": tp / (tp + fn) if (tp + fn) else None,
        "specificity_non_glaucoma": tn / (tn + fp) if (tn + fp) else None,
        "confusion": {"tp": tp, "tn": tn, "fp": fp, "fn": fn},
    }

def save_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")

## 9. Evaluar adapter QLoRA como clasificador

In [ ]:
qlora_results = []
for idx, row in enumerate(eval_rows, start=1):
    print(f"[{idx}/{len(eval_rows)}]", row["label"], row["image"])
    raw_text = generate_with_model(
        model,
        processor,
        CLASSIFICATION_PROMPT,
        image_path=row["image"],
        max_new_tokens=32,
    )
    pred_label = parse_glaucoma_label(raw_text)
    item = {
        "image": row["image"],
        "true_label": row["label"],
        "raw_text": raw_text,
        "pred_label": pred_label,
        "correct": pred_label == row["label"],
    }
    qlora_results.append(item)
    print("raw:", raw_text)
    print("pred:", pred_label, "correct:", item["correct"])

qlora_metrics = compute_binary_metrics(qlora_results)
qlora_metrics

In [ ]:
save_json("results/classification_qlora_acrima.json", {
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "model_id": MODEL_ID,
    "adapter_dir": QLORA_OUTPUT_DIR,
    "task": "glaucoma_vs_non_glaucoma",
    "train_per_class": TRAIN_PER_CLASS,
    "eval_per_class": EVAL_PER_CLASS,
    "metrics": qlora_metrics,
    "results": qlora_results,
})

print(json.dumps(qlora_metrics, indent=2))

## 10. Prueba descriptiva con adapter

Este adapter fue entrenado para clasificación, no descripción. Esta celda es solo para ver si el adapter cambió el estilo de respuesta. No es evaluación clínica.

In [ ]:
description_prompt = "Describe the ophthalmological findings in this fundus image."
description_text = generate_with_model(
    model,
    processor,
    description_prompt,
    image_path=TEST_IMAGE,
    max_new_tokens=256,
)
print(description_text)

save_json("results/description_qlora_classifier_adapter.json", {
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "model_id": MODEL_ID,
    "adapter_dir": QLORA_OUTPUT_DIR,
    "image": TEST_IMAGE,
    "prompt": description_prompt,
    "text": description_text,
    "note": "Adapter trained on ACRIMA classification labels, not expert descriptions.",
})

## 11. Preparado para QLoRA descriptivo con PubMed-Ophtha

Cuando quieras hacer QLoRA de descripción, cambia el dataset de entrenamiento para que cada ejemplo tenga:

```python
{"image": image_path, "prompt": DESCRIPTION_PROMPT, "answer": caption_or_report}
```

PubMed-Ophtha puede servir como puente image-caption. El dataset oficial experto debería reemplazarlo después. No conviene mezclar ahora clasificación ACRIMA y descripción PubMed-Ophtha en el mismo adapter si quieres interpretar resultados claramente.